# VERA — CALVIN Ablation Training
**9 ablations × 3 seeds on CALVIN D→D**

### Before running:
1. `Runtime → Change runtime type → T4 GPU` (free) or A100 (Pro)
2. Run cells top to bottom — dataset downloads automatically inside Colab (first time only, saves to Drive)

### If session dies mid-run:
- Checkpoints are saved to Drive after every epoch — nothing is lost
- Re-run all setup cells (dataset skips re-download if already on Drive)
- In Cell 7, set `START_FROM` to the ablation index that was running when it died

In [1]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:  ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

GPU available: True
Device: Tesla T4
VRAM:   15.6 GB


In [2]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints and results will be saved here — survives session crashes
DRIVE_ROOT = '/content/drive/MyDrive/VERA_CALVIN'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted. Output dir:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted. Output dir: /content/drive/MyDrive/VERA_CALVIN


In [3]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# torch / numpy / PIL already in Colab — only need CLIP and ftfy
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ftfy regex

import clip, numpy, PIL, yaml
print('clip:', clip.__version__ if hasattr(clip,'__version__') else 'ok')
print('torch:', torch.__version__)
print('All dependencies ready')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
clip: ok
torch: 2.10.0+cu128
All dependencies ready


In [4]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR  = '/content/RLConditionedVLA'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!ls

Cloning into '/content/RLConditionedVLA'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 172 (delta 10), reused 10 (delta 3), pack-reused 147 (from 1)
Receiving objects: 100% (172/172), 41.99 MiB | 26.84 MiB/s, done.
Resolving deltas: 100% (59/59), done.
Working dir: /content/RLConditionedVLA
configs  envs	     README.md	     requirements.txt  VERA_CALVIN_Colab.ipynb
data	 evaluation  references.bib  scripts
docs	 models      REPORT.md	     training


In [5]:
# ── Cell 5: Download + extract CALVIN dataset ────────────────────────────────
# Uses aria2c with 16 parallel connections — typically 5-10x faster than wget
# on slow academic servers. Saves zip to Drive so future sessions skip download.
# Extracts to local disk for fast I/O during training.
import os
CALVIN_ZIP     = f'{DRIVE_ROOT}/task_D_D.zip'    # persisted on Drive
CALVIN_EXTRACT = '/content/calvin_data'           # local disk — fast I/O
CALVIN_PATH    = f'{CALVIN_EXTRACT}/task_D_D'
CALVIN_URL     = 'http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip'
# ── Step 1: Download ──────────────────────────────────────────────────────────
if os.path.exists(CALVIN_ZIP) and os.path.getsize(CALVIN_ZIP) > 1e9:
    print(f'ZIP already on Drive ({os.path.getsize(CALVIN_ZIP)/1e9:.1f} GB) — skipping download')
else:
    # Install aria2c — opens 16 parallel connections to the server
    print('Installing aria2c...')
    !apt-get install -q -y aria2
    print(f'Downloading CALVIN task_D_D with 16 parallel connections...')
    print(f'Source: {CALVIN_URL}')
    print('(was 2.5 hrs with wget — should be ~15-30 min with aria2c)\n')
    !aria2c \
        --max-connection-per-server=16 \
        --split=16 \
        --min-split-size=1M \
        --max-concurrent-downloads=1 \
        --file-allocation=none \
        --console-log-level=notice \
        --summary-interval=30 \
        -o "task_D_D.zip" \
        -d "{DRIVE_ROOT}" \
        "{CALVIN_URL}"
    size_gb = os.path.getsize(CALVIN_ZIP) / 1e9
    print(f'\nDownload complete ({size_gb:.1f} GB)')
# ── Step 2: Extract to local disk ────────────────────────────────────────────
if os.path.exists(CALVIN_PATH + '/training'):
    print('Already extracted — skipping')
else:
    print('Extracting to local disk (~3 min)...')
    os.makedirs(CALVIN_EXTRACT, exist_ok=True)
    !unzip -q "{CALVIN_ZIP}" -d "{CALVIN_EXTRACT}"
    print('Extraction complete')
    !ls {CALVIN_PATH}
# ── Step 3: Verify ────────────────────────────────────────────────────────────
assert os.path.exists(CALVIN_PATH + '/training'),   'training/ missing'
assert os.path.exists(CALVIN_PATH + '/validation'), 'validation/ missing'
ann = CALVIN_PATH + '/training/lang_annotations/auto_lang_ann.npy'
assert os.path.exists(ann), f'lang_annotations missing at {ann}'
print('CALVIN dataset OK:', CALVIN_PATH)

Source: http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip
VIN/task_D_D.zip      6%[>                   ]  10.89G  20.2MB/s    eta 2h 17m ^C
Download complete (11.7 GB)
Extracting to local disk (~3 min)...
[/content/drive/MyDrive/VERA_CALVIN/task_D_D.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/drive/MyDrive/VERA_CALVIN/task_D_D.zip or
        /content/drive/MyDrive/VERA_CALVIN/task_D_D.zip.zip, and cannot find /content/drive/MyDrive/VERA_CALVIN/task_D_D.zip.ZIP, period.
Extraction complete
ls: cannot access '/content/calvin_data/task_D_D': No such file or directory


AssertionError: training/ missing — extraction failed

In [ ]:
# ── Cell 6: Smoke test — verify loader works before full run ─────────────────
import sys
sys.path.insert(0, REPO_DIR)

from data.trajectory_dataset import load_calvin

print('Loading a few CALVIN episodes to verify loader...')
eps = load_calvin(CALVIN_PATH, split='training')

print(f'Episodes loaded: {len(eps)}')
print(f'Keys: {list(eps[0].keys())}')
print(f'Frames shape:  {eps[0]["frames"].shape}')
print(f'Actions shape: {eps[0]["actions"].shape}  dtype={eps[0]["actions"].dtype}')
print(f'Action range:  {eps[0]["actions"].min()} – {eps[0]["actions"].max()}  (expect 0–13)')
print(f'Action vectors:{eps[0]["action_vectors"].shape}  (expect T×7)')
print(f'Instruction:   {eps[0]["instruction"]}')
print('Loader OK')

In [ ]:
# ── Cell 7: Configure run ─────────────────────────────────────────────────────
# ↓ ONLY change these two values ↓

START_FROM = 0    # Resume from ablation index if session died (0 = start fresh)
                  # Indices: 0=full_vera  1=bc_baseline  2=no_exp  3=no_act
                  #          4=no_lang    5=no_history_tf 6=no_reward_gate
                  #          7=no_dual_head  8=corrupted_conseq

DRY_RUN = False   # Set True first to verify (2 epochs, synthetic data, ~2 min)
                  # Set False for the real run

# ─────────────────────────────────────────────────────────────────────────────
# Point checkpoints to Drive so they survive session crashes
import yaml, os

cfg_path = f'{REPO_DIR}/configs/calvin_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Redirect output to Drive
cfg['data']['episodes_path']   = CALVIN_PATH
cfg['training']['output_dir']  = f'{DRIVE_ROOT}/checkpoints'
cfg['training']['device']      = 'cuda'
cfg['training']['num_workers'] = 2    # Colab works better with 2

# Save patched config
colab_cfg_path = f'{REPO_DIR}/configs/calvin_colab_runtime.yaml'
with open(colab_cfg_path, 'w') as f:
    yaml.dump(cfg, f)

print('Config ready')
print(f'  Dataset:  {CALVIN_PATH}')
print(f'  Output:   {DRIVE_ROOT}/checkpoints')
print(f'  Epochs:   {cfg["training"]["epochs"]} (max, early stopping patience=10)')
print(f'  Ablations to run: {9 - START_FROM} (starting from index {START_FROM})')
print(f'  Seeds: 42, 123, 456')
print(f'  DRY RUN: {DRY_RUN}')

In [ ]:
# ── Cell 8: Run ablations ─────────────────────────────────────────────────────
# This cell runs all 9 ablations × 3 seeds sequentially.
# Checkpoints save to Drive after every epoch — safe to interrupt and resume.
# Expected time on T4:  ~20-40 min per (ablation × seed)  →  ~9-18 hours total
# Expected time on A100: ~8-15 min per (ablation × seed)  →  ~4-7  hours total

import subprocess, sys

cmd = [
    sys.executable,
    f'{REPO_DIR}/scripts/run_calvin_ablations.py',
    '--calvin_path', CALVIN_PATH,
    '--config',      colab_cfg_path,
    '--out',         f'{DRIVE_ROOT}/checkpoints',
    '--start_from',  str(START_FROM),
]
if DRY_RUN:
    cmd.append('--dry_run')

print('Running:', ' '.join(cmd))
print('='*65)

# Run with live output
result = subprocess.run(cmd, cwd=REPO_DIR)
print('='*65)
print('Exit code:', result.returncode)

In [ ]:
# ── Cell 9: View results ──────────────────────────────────────────────────────
import json, numpy as np

results_path = f'{DRIVE_ROOT}/checkpoints/calvin_ablation_summary.json'

with open(results_path) as f:
    results = json.load(f)

print(f'  {"Ablation":<46} {"Mean":>7}  {"Std":>6}  Seeds')
print(f'  {"-"*46} {"-"*7}  {"-"*6}  {"-"*22}')
for slug, v in results.items():
    seeds = '  '.join(f'{x:.4f}' for x in v['seeds'].values())
    print(f'  {v["display"]:<46} {v["mean_val_acc"]:.4f}  {v["std_val_acc"]:.4f}  {seeds}')

In [ ]:
# ── Cell 10: Download results JSON to local machine ───────────────────────────
from google.colab import files
import shutil, os

# Bundle the summary JSON + all training logs into one zip
zip_path = '/content/calvin_results.zip'
shutil.make_archive(
    '/content/calvin_results', 'zip',
    f'{DRIVE_ROOT}/checkpoints'
)
files.download(zip_path)
print('Download started')